# 01 — Inventory, linkage, QC and shortcut audit
### Tongue-AI × Atrophic Gastritis · UMC HCMC cohort (n = 1,226)

**No GPU needed.** Runtime → Change runtime type → **CPU** is fine (and starts faster).

This notebook is the Phase-2 gate of the project harness. It answers four questions
before anyone trains anything:

1. How many images exist, and does every one map to a patient in `Data.csv`?
2. Which patients have no image, and which images have no patient?
3. Were the two arms photographed in **different periods or on different devices**?
4. Can the label be predicted **without looking at the tongue** — from metadata or
   from the image border alone?

Question 4 is the one that decides whether this project has a paper or a problem.
If the answer is yes, the training notebook is not written until the design is fixed.

---

### Before you run

The folder is **shared with you**, not owned by you, so it will not appear under
`MyDrive` until you add a shortcut:

> Open **ĐỀ TÀI LƯỠI AI** in Drive → right-click → **Organise → Add shortcut to Drive** → put it in **My Drive**

Then upload `Data.csv` when the notebook asks, or drop it into your working folder first.

### What to send back

The last cell prints a compact report. **Copy that whole block back into the chat.**
Do not paste images, tracebacks in full, or the raw manifest.

In [ ]:
#@title 1 · Install and mount
!pip -q install pillow-heif exifread > /dev/null 2>&1

import os, sys, json, re, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

import pillow_heif, PIL, pandas as pd, numpy as np, sklearn
pillow_heif.register_heif_opener()

ENV = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__, "numpy": np.__version__,
    "sklearn": sklearn.__version__, "Pillow": PIL.__version__,
    "pillow_heif": pillow_heif.__version__,
}
print("environment:", json.dumps(ENV, indent=2))

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }

# Path to the SHARED folder after you added the shortcut to My Drive.
SHARED_ROOT = "/content/drive/MyDrive/\u0110\u1ec0 T\u00c0I L\u01af\u1ee0I AI"  #@param {type:"string"}

# Your own working folder. Derived artefacts go here; the shared folder is never written to.
WORK_ROOT   = "/content/drive/MyDrive/tongue-ai-work"  #@param {type:"string"}

RESIZE_PX   = 512   #@param {type:"integer"}
SEED        = 20260915  #@param {type:"integer"}

SHARED = Path(SHARED_ROOT); WORK = Path(WORK_ROOT)
PNG_DIR = WORK / f"png_{RESIZE_PX}"; REPORT_DIR = WORK / "reports"
for d in (WORK, PNG_DIR, REPORT_DIR): d.mkdir(parents=True, exist_ok=True)

assert SHARED.is_dir(), (
    f"Not found: {SHARED}\n"
    "Did you add a shortcut to My Drive? Drive > right-click the folder > "
    "Organise > Add shortcut to Drive > My Drive."
)
print("shared :", SHARED)
print("work   :", WORK)
print("\ntop level of shared folder:")
for p in sorted(SHARED.iterdir()):
    print("  ", p.name, "/" if p.is_dir() else "")

In [ ]:
#@title 3 · Inventory — walk the tree and parse every filename

# Rule: H{Patient_ID}{k}.HEIC, trailing digit k is the photo index.
#   k in {1,2} -> dorsal, k == 3 -> sublingual.  H12672 -> patient 1267, dorsal #2.
# The rule is INFERRED from samples, so every parse is validated against Data.csv
# in the next cell and failures are quarantined, never dropped.

IMAGE_EXT = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
VIDEO_EXT = {".mov", ".mp4", ".m4v", ".avi"}
VIEW_BY_INDEX = {"1": "dorsal", "2": "dorsal", "3": "sublingual"}
FNAME_RE = re.compile(r"^H(\d{2,6})$", re.IGNORECASE)

def arm_from_path(p: Path):
    s = " ".join(x.upper() for x in p.parts)
    if "VKTDD" in s: return "non_atrophic"
    if "VTDD"  in s: return "atrophic"
    return ""

def read_exif(p: Path):
    out = {"exif_datetime": "", "exif_make": "", "exif_model": ""}
    try:
        import exifread
        with p.open("rb") as fh:
            t = exifread.process_file(fh, details=False, stop_tag="EXIF DateTimeOriginal")
        out["exif_datetime"] = str(t.get("EXIF DateTimeOriginal", ""))
        out["exif_make"]     = str(t.get("Image Make", ""))
        out["exif_model"]    = str(t.get("Image Model", ""))
    except Exception:
        pass
    return out

REWALK = False  #@param {type:"boolean"}
MANIFEST = WORK / "manifest.csv"

# The walk takes ~3 minutes and a Colab runtime can be recycled at any time.
# Caching the manifest to Drive makes recovery after a disconnect instant:
# re-run cells 1, 2, 3, 4 and this cell reloads instead of re-walking.
if MANIFEST.exists() and not REWALK:
    man = pd.read_csv(MANIFEST)
    print(f"loaded cached manifest: {len(man)} rows  (tick REWALK to force a fresh walk)")
else:
    rows, t0 = [], time.time()
    for p in SHARED.rglob("*"):
        if not p.is_file(): continue
        ext = p.suffix.lower()
        kind = "image" if ext in IMAGE_EXT else ("video" if ext in VIDEO_EXT else None)
        if kind is None: continue
        m = FNAME_RE.match(p.stem.strip())
        pid = idx = view = None; ok = False; reason = ""
        if not m:
            reason = "does not match H<digits>"
        else:
            digits = m.group(1); pid, idx = int(digits[:-1]) if digits[:-1] else None, digits[-1]
            view = VIEW_BY_INDEX.get(idx)
            if pid is None:   reason = "no patient digits before index"
            elif view is None: reason = f"unexpected index digit {idx}"
            else:              ok = True
        ex = read_exif(p) if kind == "image" else {"exif_datetime": "", "exif_make": "", "exif_model": ""}
        st = p.stat()
        rows.append(dict(path=str(p), filename=p.name, kind=kind, patient_id=pid,
                         image_index=idx or "", view=view or "", parse_ok=int(ok),
                         parse_reason=reason, arm_folder=arm_from_path(p),
                         bytes=st.st_size, mtime=st.st_mtime, **ex))
    man = pd.DataFrame(rows)
    man.to_csv(MANIFEST, index=False)
    print(f"walked in {time.time()-t0:.1f}s")
print(f"  files indexed : {len(man)}")
print(f"  images        : {(man.kind=='image').sum()}")
print(f"  videos        : {(man.kind=='video').sum()}")
print(f"  unparseable   : {(man.parse_ok==0).sum()}")
print()
print(man[man.kind=='image'].groupby(['arm_folder','view']).size().to_frame('n'))
if (man.parse_ok==0).any():
    print("\nunparseable examples:")
    print(man[man.parse_ok==0][['filename','parse_reason']].head(10).to_string(index=False))

In [ ]:
#@title 4 · Load Data.csv and reconcile

import shutil

csv_path = WORK / "Data.csv"
if not csv_path.exists():
    local = Path("Data.csv")
    if not local.exists():
        print("Data.csv not found — upload it now.")
        from google.colab import files
        up = files.upload()
        local = Path(list(up)[0])
    # copy, never rename: /content is a local disk and /content/drive is a FUSE
    # mount, so os.replace across them raises OSError 18 (cross-device link).
    shutil.copy(local, csv_path)
    print(f"saved to {csv_path}")

cohort = pd.read_csv(csv_path, encoding="utf-8-sig")
cohort["arm_csv"] = cohort.Atrophic_Gastritis.map({1: "atrophic", 0: "non_atrophic"})
print("Data.csv:", cohort.shape)

img = man[(man.kind=="image") & (man.parse_ok==1)].copy()
img["patient_id"] = img.patient_id.astype(int)
known = set(cohort.Patient_ID)
img["in_cohort"] = img.patient_id.isin(known)

per_patient = (img[img.in_cohort]
               .pivot_table(index="patient_id", columns="view", values="filename", aggfunc="count")
               .fillna(0).astype(int))
for c in ("dorsal","sublingual"):
    if c not in per_patient: per_patient[c] = 0

covered  = set(per_patient.index)
missing  = sorted(known - covered)
orphan   = sorted(set(img.loc[~img.in_cohort, "patient_id"]))

lab = img[img.in_cohort].merge(cohort[["Patient_ID","arm_csv"]],
                               left_on="patient_id", right_on="Patient_ID")
lab_f = lab[lab.arm_folder != ""]
mismatch = sorted(set(lab_f.loc[lab_f.arm_folder != lab_f.arm_csv, "patient_id"]))

LINK = dict(
    n_cohort=len(cohort), n_images=int((man.kind=="image").sum()), n_parsed=len(img),
    n_unparsed=int((man.kind=="image").sum()-len(img)),
    patients_with_images=len(covered), patients_without_images=len(missing),
    orphan_image_ids=len(orphan), label_disagreements=len(mismatch),
)
print(json.dumps(LINK, indent=2))
print("\nimages per patient (count of patients):")
print(per_patient[["dorsal","sublingual"]].apply(pd.Series.value_counts).fillna(0).astype(int))
if mismatch:
    print(f"\n!! folder arm disagrees with Atrophic_Gastritis for {len(mismatch)} patients")
    print("   ", mismatch[:40])
if orphan:
    print(f"\n!! {len(orphan)} image patient-ids are absent from Data.csv (QUARANTINE)")
    print("   ", orphan[:40])

In [ ]:
#@title 5 · Acquisition-confound QC — were the two arms shot differently?

d = img[img.in_cohort].merge(cohort[["Patient_ID","arm_csv","Atrophic_Gastritis"]],
                             left_on="patient_id", right_on="Patient_ID")
d["mtime_month"] = pd.to_datetime(d.mtime, unit="s").dt.to_period("M").astype(str)

print("=== file mtime month x arm ===")
mt = pd.crosstab(d.mtime_month, d.arm_csv); print(mt)

exif_ok = d[d.exif_datetime.astype(str).str.len() > 0].copy()
print(f"\n=== EXIF capture month x arm  (EXIF present on {len(exif_ok)}/{len(d)} images) ===")
if len(exif_ok):
    exif_ok["exif_month"] = pd.to_datetime(exif_ok.exif_datetime, format="%Y:%m:%d %H:%M:%S",
                                           errors="coerce").dt.to_period("M").astype(str)
    et = pd.crosstab(exif_ok.exif_month, exif_ok.arm_csv); print(et)
else:
    et = None
    print("no EXIF timestamps readable — Drive sync may have stripped them")

print("\n=== camera model x arm ===")
if (d.exif_model.astype(str).str.len() > 0).any():
    cm = pd.crosstab(d.exif_model, d.arm_csv); print(cm)
else:
    cm = None; print("no camera model in EXIF")

print("\n=== file size (bytes) x arm ===")
sz = d.groupby("arm_csv").bytes.describe(); print(sz)
print("\nA systematic size or device difference between arms is itself a shortcut.")

In [ ]:
#@title 6 · Convert HEIC to PNG once, cache to Drive   (the slow cell)

from PIL import Image
from concurrent.futures import ThreadPoolExecutor

def convert(rec):
    src = Path(rec["path"]); view = rec["view"] or "unknown"
    dst = PNG_DIR / view / f"{src.stem}.png"
    if dst.exists():
        return {"src": src.name, "status": "cached", "reason": ""}
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(src) as im:
            im = im.convert("RGB")           # re-encoding here also drops all EXIF
            im.thumbnail((RESIZE_PX, RESIZE_PX), Image.LANCZOS)
            im.save(dst, format="PNG", optimize=True)
        return {"src": src.name, "status": "ok", "reason": ""}
    except Exception as e:
        return {"src": src.name, "status": "fail", "reason": f"{type(e).__name__}: {e}"}

todo = img.to_dict("records")
print(f"converting {len(todo)} images -> {PNG_DIR}  (resume-safe: re-run after a disconnect)")
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    conv = list(ex.map(convert, todo))
conv = pd.DataFrame(conv)
conv.to_csv(WORK / "convert_log.csv", index=False)

CONV = conv.status.value_counts().to_dict()
print(f"done in {time.time()-t0:.0f}s: {CONV}")
if (conv.status=="fail").any():
    print(conv[conv.status=="fail"].head(10).to_string(index=False))

In [ ]:
#@title 7 · Shortcut audit — can the label be predicted WITHOUT the tongue?

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

def verdict(a):
    if a >= 0.80: return "SEVERE - label recoverable from acquisition alone"
    if a >= 0.60: return "CONFOUNDED - a real shortcut is present"
    return "clean"

def grouped_auc(X, y, groups):
    cv  = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    p   = cross_val_predict(clf, X, y, groups=groups, cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, p))

VIEW = "dorsal"
dv = d[d.view == VIEW].reset_index(drop=True)
y, g = dv.Atrophic_Gastritis.to_numpy(), dv.patient_id.to_numpy()
AUDIT = {}

AUDIT["file_metadata"] = grouped_auc(dv[["bytes","mtime"]].to_numpy(), y, g)

ex = dv[dv.exif_datetime.astype(str).str.len() > 0]
if len(ex) > 50:
    ts = (pd.to_datetime(ex.exif_datetime, format="%Y:%m:%d %H:%M:%S", errors="coerce")
            .astype("int64") / 1e9).fillna(0).to_numpy()
    mc = ex.exif_model.astype("category").cat.codes.to_numpy()
    AUDIT["exif"] = grouped_auc(np.column_stack([ts, mc]),
                                ex.Atrophic_Gastritis.to_numpy(), ex.patient_id.to_numpy())

def border_feats(p, band=16):
    with Image.open(p) as im:
        a = np.asarray(im.convert("RGB").resize((160,160)), dtype=np.float32)
    b = np.concatenate([a[:band].reshape(-1,3), a[-band:].reshape(-1,3),
                        a[:,:band].reshape(-1,3), a[:,-band:].reshape(-1,3)])
    return np.concatenate([b.mean(0), b.std(0)])

feats, keep = [], []
for i, r in dv.iterrows():
    p = PNG_DIR / VIEW / f"{Path(r.path).stem}.png"
    if p.exists():
        try: feats.append(border_feats(p)); keep.append(i)
        except Exception: pass
if len(keep) > 50:
    AUDIT["image_border"] = grouped_auc(np.vstack(feats), y[keep], g[keep])

print(f"{'channel':<16} {'AUC':>6}   verdict")
for k, v in AUDIT.items():
    print(f"{k:<16} {v:>6.3f}   {verdict(v)}")

GATE = "PASS" if all(v < 0.60 for v in AUDIT.values()) else "FAIL"
print(f"\nGATE: {GATE}")
if GATE == "FAIL":
    print("Do not train on these images as-is. Options:")
    print("  (a) match/stratify arms on acquisition period, re-run")
    print("  (b) train within-period, report period-held-out as primary")
    print("  (c) reframe the paper around the confound - honest and publishable")

In [ ]:
#@title 8 · Final report — COPY THIS BLOCK BACK INTO THE CHAT

report = {
    "notebook": "01_inventory_qc",
    "generated_utc": pd.Timestamp.utcnow().isoformat(),
    "environment": ENV,
    "seed": SEED,
    "linkage": LINK,
    "images_per_patient": {
        c: per_patient[c].value_counts().sort_index().to_dict() for c in ("dorsal","sublingual")
    },
    "by_arm_view": img[img.in_cohort].merge(
        cohort[["Patient_ID","arm_csv"]], left_on="patient_id", right_on="Patient_ID"
    ).groupby(["arm_csv","view"]).size().to_dict().__str__(),
    "conversion": CONV,
    "mtime_month_by_arm": mt.to_dict(),
    "exif_month_by_arm": (et.to_dict() if et is not None else None),
    "camera_model_by_arm": (cm.to_dict() if cm is not None else None),
    "bytes_by_arm": sz.round(1).to_dict(),
    "shortcut_audit": {k: round(v, 4) for k, v in AUDIT.items()},
    "gate": GATE,
}
out = REPORT_DIR / "01_report.json"
out.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

print("=" * 66)
print("  PASTE EVERYTHING BELOW THIS LINE BACK INTO THE CHAT")
print("=" * 66)
print(json.dumps(report, indent=2, default=str))
print("=" * 66)
print(f"also saved to {out}")

---
## What happens next

- **GATE: PASS** → the training notebook `02_train_image.ipynb` gets written, and it
  will reuse the `png_512/` cache this notebook just built. No re-decoding.
- **GATE: FAIL** → do not train yet. The result would measure the camera and the
  calendar rather than the patient. Bring the numbers back and the study design gets
  adjusted first.

Either way, paste the report block from cell 8 into the chat.